In [ ]:
import requests

# Coordinates are (longitude, latitude)
source = "88.3639,22.5726"      # Kolkata Example
destination = "88.4352,22.5851" # Salt Lake Example

url = f"http://router.project-osrm.org/route/v1/driving/{source};{destination}"
params = {
    "alternatives": "true", # Get alternative routes
    "geometries": "geojson",
    "overview": "full"
}

response = requests.get(url, params=params)
data = response.json()

if data['code'] == 'Ok':
    print(f"Found {len(data['routes'])} routes:")
    for i, route in enumerate(data['routes']):
        distance = route['distance'] / 1000 # Convert to km
        duration = route['duration'] / 60   # Convert to minutes
        print(f"Route {i+1}: {distance:.2f} km, {duration:.2f} min")

Found 2 routes:
Route 1: 10.11 km, 12.45 min
Route 2: 9.72 km, 12.93 min


In [10]:
data['routes'][0]

{'legs': [{'steps': [],
   'weight': 757.3,
   'summary': '',
   'duration': 747,
   'distance': 10105}],
 'weight_name': 'routability',
 'geometry': {'coordinates': [[88.364017, 22.57286],
   [88.363849, 22.572937],
   [88.36374, 22.572991],
   [88.363874, 22.573238],
   [88.363939, 22.573373],
   [88.364033, 22.573345],
   [88.364295, 22.573233],
   [88.364527, 22.573712],
   [88.364811, 22.573569],
   [88.364821, 22.573566],
   [88.365421, 22.573358],
   [88.365685, 22.57327],
   [88.36614, 22.573077],
   [88.366509, 22.572973],
   [88.366997, 22.572788],
   [88.36705, 22.57277],
   [88.367769, 22.572485],
   [88.367924, 22.572443],
   [88.368372, 22.571787],
   [88.368467, 22.571647],
   [88.368618, 22.571452],
   [88.368868, 22.571012],
   [88.369059, 22.570662],
   [88.369399, 22.570038],
   [88.36941, 22.570019],
   [88.370149, 22.568757],
   [88.370199, 22.568673],
   [88.370148, 22.568576],
   [88.369503, 22.567259],
   [88.369488, 22.567226],
   [88.369081, 22.566321],
   [88

In [11]:
import requests

# Coordinates for Kolkata
lat, lon = 22.57, 88.36
url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"

response = requests.get(url)
data = response.json()

temp = data['current_weather']['temperature']
print(f"The current temperature is {temp}°C")

The current temperature is 31.2°C


In [2]:
import os,mlflow
os.environ['MLFLOW_TRACKING_USERNAME']='ronaksah75'
os.environ['MLFLOW_TRACKING_PASSWORD']="f0112720c5d8bfcace1909f31c54e3711481166f"
mlflow.set_tracking_uri("https://dagshub.com/ronaksah75/smart-supply-chain.mlflow")
model=mlflow.pyfunc.load_model("models:/route_predictor@champion")
scaler=mlflow.sklearn.load_model("models:/scaler_model@champion")

[2026-04-17 20:28:48,540: INFO: utils: NumExpr defaulting to 16 threads.]


d:\conda_envs\mlops\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd
def transform_df(df):

            # day time encodeing
    mapping = {'Morning':1, 'Evening':2, 'Afternoon':3, 'Night':4}
    df['time_of_day'] = df['time_of_day'].map(mapping)

                # week day encoding
    mapping_day_of_week = {'Weekday':1, 'Weekend':2}
    df['day_of_week'] = df['day_of_week'].map(mapping_day_of_week)
                
        # weather encoding
    mapping_weather = {'Clear':1, 'Rain':2, 'Heatwave':3, 'Fog':4}
    df['weather_condition'] = df['weather_condition'].map(mapping_weather)

            # road encoding
    mapping_road = {'Highway':1, 'Inner Road':2, 'Main Road':3}
    df['road_type'] = df['road_type'].map(mapping_road)

            #  traffic encodeing
    mapping_traffic = {'Low':1,'Medium':2, 'High':3, 'Very High':4}
    df['traffic_density_level'] = df['traffic_density_level'].map(mapping_traffic)

   
    X_scaled =scaler.transform(df)
    transformed_df = pd.DataFrame(X_scaled, columns=df.columns)
    return transformed_df

def predict(self,config):

        # df=pd.DataFrame({
        #     "distance_km":[self.config.distance_km],"time_of_day":[self.config.time_of_day],
        #     "day_of_week": [self.config.day_of_week],"weather_condition": [self.config.weather_condition],
        #     "traffic_density_level": [self.config.traffic_density_level],"road_type": [self.config.road_type],
        #     "average_speed_kmph": [self.config.average_speed_kmph]
        # })
    df=pd.DataFrame([config])
        

        # df=df.astype(float)
    scaled_df=transform_df(df)
    y=model.predict(scaled_df)

    return y[0]

In [4]:
config={"distance_km":10.11,"time_of_day":"Evening",
            "day_of_week": "Weekday","weather_condition": "Clear",
            "traffic_density_level": "Medium","road_type": "Highway","average_speed_kmph": 60
        }
predict(config)


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "d:\conda_envs\mlops\lib\site-packages\IPython\core\interactiveshell.py", line 3579, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\ronak\AppData\Local\Temp\ipykernel_11068\1186415735.py", line 5, in <module>
    predict(config)
TypeError: predict() missing 1 required positional argument: 'config'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "d:\conda_envs\mlops\lib\site-packages\IPython\core\interactiveshell.py", line 2170, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
  File "d:\conda_envs\mlops\lib\site-packages\IPython\core\ultratb.py", line 1457, in structured_traceback
    return FormattedTB.structured_traceback(
  File "d:\conda_envs\mlops\lib\site-packages\IPython\core\ultratb.py", line 1348, in structured_traceback
    return VerboseTB.structured_traceback(
  File "d:\conda_envs\mlops\lib\site-packages\IPython\co